# List Comprehensions — Lesson

List comprehensions are **the most Pythonic way to build a list from another iterable**. They replace verbose `for`-loop + `append` patterns with a single, readable expression. Mastering them is non-negotiable: they appear in nearly every interview, every backend codebase, and every data pipeline.

### Why this matters
- **Interviews:** Cleaner code in less time. Interviewers expect comprehensions over manual loops for transforms/filters. They also test whether you know when *not* to use them.
- **Backend:** Building response payloads, filtering query results, transforming model objects — all comprehension patterns.
- **Data engineering:** Cleaning records, projecting columns, exploding nested structures.
- **Performance:** Comprehensions are faster than equivalent `for` + `append` loops because the append is implemented in C.

### What you'll learn
1. The mental model — comprehensions as expressions
2. Basic list comprehension syntax
3. Filtering with `if`
4. Transforming with conditional expressions (`if/else`)
5. Nested loops in comprehensions
6. Nested list comprehensions (matrices / 2D)
7. Set comprehensions
8. Dict comprehensions
9. Generator expressions (and when to prefer them)
10. Comprehensions with `enumerate`, `zip`, `range`
11. Multiple `if` filters and the walrus operator
12. Common interview patterns: map / filter / flatten / transpose / dedupe / group
13. Performance and memory considerations
14. Readability — when *not* to use a comprehension
15. Gotchas and anti-patterns

By the end you should be able to read any comprehension at a glance and write the right one without thinking.

---
## 1. The Mental Model

A list comprehension is an **expression that produces a list**. Read it left-to-right:

```
[ <expression>  for <var> in <iterable>  if <condition> ]
   ^what to put   ^how to get each item    ^optional filter
```

Equivalent imperative loop:

```python
result = []
for var in iterable:
    if condition:
        result.append(expression)
```

Three things to internalise:
- The **expression** is what ends up in the list (it can use `var`).
- The **`for` clause** drives iteration.
- The **`if` clause** is a filter — items that fail it are dropped.

If you can mentally translate any comprehension into the imperative form above, you understand them.

In [ ]:
# Imperative loop
squares = []
for n in range(6):
    squares.append(n * n)
print(squares)

# Same thing as a comprehension
squares = [n * n for n in range(6)]
print(squares)

---
## 2. Basic List Comprehension

The simplest form: transform every item in an iterable.

In [ ]:
# Square each number
nums = [1, 2, 3, 4, 5]
squares = [n ** 2 for n in nums]
print(squares)  # [1, 4, 9, 16, 25]

# Uppercase each string
words = ["hello", "world", "python"]
upper = [w.upper() for w in words]
print(upper)

# Length of each string
lengths = [len(w) for w in words]
print(lengths)

# Build a list of constant values
zeros = [0 for _ in range(5)]
print(zeros)

**Tip:** Use `_` as the loop variable when you don't need the value (e.g. `[0 for _ in range(n)]` to build a zero-filled list).

---
## 3. Filtering with `if`

Add an `if` clause **after** the `for` clause to drop items.

```
[ expr for x in iterable if condition ]
```

The `if` here is a **filter**, not a transformation. Only items where `condition` is truthy reach the expression.

In [ ]:
nums = range(20)

# Keep only even numbers
evens = [n for n in nums if n % 2 == 0]
print(evens)

# Keep words longer than 4 chars
words = ["hi", "hello", "yo", "python", "go"]
long_words = [w for w in words if len(w) > 4]
print(long_words)

# Filter out None / falsy values (very common pattern)
data = [1, None, 2, "", 3, 0, 4]
truthy = [x for x in data if x]
print(truthy)

### Multiple filters
You can chain `if` clauses — they are AND'd together.

In [ ]:
# Numbers that are even AND divisible by 3
nums = range(30)
result = [n for n in nums if n % 2 == 0 if n % 3 == 0]
print(result)

# Equivalent and usually preferred:
result2 = [n for n in nums if n % 2 == 0 and n % 3 == 0]
print(result2)

---
## 4. Conditional Expression (`if / else`) — Transform vs Filter

This is the **#1 source of confusion**. There are two `if`s:

| Position | Role | Syntax |
|----------|------|--------|
| **Before `for`** | Transform — picks the value | `[a if cond else b for x in xs]` |
| **After `for`** | Filter — picks which items go through | `[x for x in xs if cond]` |

You can also combine both.

In [ ]:
nums = range(-5, 6)

# Transform: replace negatives with 0 (KEEP every item)
clamped = [n if n >= 0 else 0 for n in nums]
print(clamped)

# Filter: keep only non-negatives (DROP some items)
non_neg = [n for n in nums if n >= 0]
print(non_neg)

# Combine: filter first, then transform
# Square the non-negatives only
result = [n ** 2 for n in nums if n >= 0]
print(result)

# Combine: transform AND filter
# For numbers > 0: keep n; otherwise replace with None.
# Then drop the Nones (two-step).
labelled = [n if n > 0 else None for n in nums]
print(labelled)
filtered = [n for n in labelled if n is not None]
print(filtered)

**Memorise this rule of thumb:**
- *"if I want to keep everything but change some values"* → `expr if cond else other_expr` **before** `for`.
- *"if I want to drop some items"* → `if cond` **after** `for`.

---
## 5. Nested Loops in Comprehensions

You can stack multiple `for` clauses. They read **left-to-right, outermost first** — same order as nested loops.

```python
[ expr for x in xs for y in ys ]

# is equivalent to:
result = []
for x in xs:
    for y in ys:
        result.append(expr)
```

In [ ]:
# All (row, col) pairs of a 3x3 grid
pairs = [(r, c) for r in range(3) for c in range(3)]
print(pairs)

# Cartesian product of two lists
colors = ["red", "blue"]
sizes = ["S", "M", "L"]
combos = [f"{c}-{s}" for c in colors for s in sizes]
print(combos)

# Flatten a 2D list (very common interview pattern)
matrix = [[1, 2, 3], [4, 5, 6], [7, 8, 9]]
flat = [x for row in matrix for x in row]
print(flat)

### Inner loop can depend on outer loop
The right-hand iterable can use the variable bound on the left.

In [ ]:
# Triangular pairs (j > i) — useful for pair-iteration without duplicates
n = 4
pairs = [(i, j) for i in range(n) for j in range(i + 1, n)]
print(pairs)

---
## 6. Nested *List* Comprehensions (a list of lists)

Different from nested loops! Here the **expression itself** is a comprehension. This is how you build matrices.

In [ ]:
# 3x3 matrix of zeros
matrix = [[0 for _ in range(3)] for _ in range(3)]
for row in matrix:
    print(row)

# Identity matrix
n = 4
identity = [[1 if r == c else 0 for c in range(n)] for r in range(n)]
for row in identity:
    print(row)

# Multiplication table
table = [[r * c for c in range(1, 6)] for r in range(1, 6)]
for row in table:
    print(row)

### ⚠️ Critical gotcha: do NOT use `* n` to make a 2D list

```python
grid = [[0] * 3] * 3   # WRONG — every row is the SAME list
```

All three "rows" are the same object. Mutating one mutates all of them. Always use a comprehension for the outer dimension.

In [ ]:
# WRONG
bad = [[0] * 3] * 3
bad[0][0] = 99
print(bad)  # [[99, 0, 0], [99, 0, 0], [99, 0, 0]] — yikes

# RIGHT
good = [[0] * 3 for _ in range(3)]
good[0][0] = 99
print(good)  # [[99, 0, 0], [0, 0, 0], [0, 0, 0]]

### Transposing a matrix
A classic comprehension idiom (and a common interview question).

In [ ]:
matrix = [
    [1, 2, 3],
    [4, 5, 6],
    [7, 8, 9],
]

# Transpose with a nested comprehension
transposed = [[row[i] for row in matrix] for i in range(len(matrix[0]))]
for row in transposed:
    print(row)

# Or use zip — more Pythonic
transposed_zip = [list(row) for row in zip(*matrix)]
print(transposed_zip)

---
## 7. Set Comprehensions

Same syntax as list comprehensions but with `{}` — produces a **set** (unique items, unordered).

In [ ]:
# Unique lengths of words
words = ["hi", "hello", "yo", "world", "go", "python"]
unique_lengths = {len(w) for w in words}
print(unique_lengths)

# Unique vowels in a sentence
sentence = "The quick brown fox jumps over the lazy dog"
vowels = {ch.lower() for ch in sentence if ch.lower() in "aeiou"}
print(vowels)

# Dedupe + transform in one step
emails = ["A@x.com", "b@X.com", "a@x.com", "B@x.COM"]
normalized = {e.lower() for e in emails}
print(normalized)

---
## 8. Dict Comprehensions

Use `{key: value for ... in ...}` — produces a `dict`.

The key/value separator (`:`) is what tells Python it's a dict comprehension and not a set comprehension.

In [ ]:
# Map number → its square
squares = {n: n ** 2 for n in range(6)}
print(squares)

# Build a frequency map (count chars in a string)
s = "mississippi"
freq = {ch: s.count(ch) for ch in set(s)}
print(freq)
# Note: Counter(s) is faster — comprehension above is O(n^2) due to .count() inside.

# Invert a dict (swap keys/values)
d = {"a": 1, "b": 2, "c": 3}
inv = {v: k for k, v in d.items()}
print(inv)

# Filter a dict by value
scores = {"alice": 92, "bob": 67, "carol": 85, "dave": 50}
passing = {name: score for name, score in scores.items() if score >= 70}
print(passing)

# Build a lookup from two parallel lists
keys = ["id", "name", "email"]
vals = [42, "Sam", "sam@x.com"]
record = {k: v for k, v in zip(keys, vals)}
print(record)

---
## 9. Generator Expressions

Same syntax as a list comprehension but with **parentheses** instead of brackets. Returns a **generator** — items are produced lazily, one at a time.

```python
gen = (n * n for n in range(1_000_000))   # builds nothing yet
```

### When to prefer a generator
- You only need to iterate **once**.
- You're aggregating (sum/min/max/any/all) — no need to materialise a list.
- The data is huge and you don't want to hold it all in memory.
- You're chaining transformations in a pipeline.

### When you need a list instead
- You need to index it (`result[3]`).
- You need its length (`len(result)`).
- You need to iterate multiple times.

In [ ]:
# Generator vs list — sum of squares
import sys

# Memory-heavy: builds the whole list
total_list = sum([n * n for n in range(1_000_000)])

# Memory-light: generator, no intermediate list
total_gen = sum(n * n for n in range(1_000_000))

print(total_list == total_gen)

# When passed as the only argument to a function, the parens can be omitted
print(sum(n for n in range(10)))   # 45
print(max(len(w) for w in ["hi", "hello", "yo"]))  # 5

**`any` / `all` short-circuit on generators** — they stop as soon as the answer is known. Pair them with a generator expression for clean, fast code.

In [ ]:
nums = [2, 4, 6, 7, 8, 10]

# Stops at 7 — never evaluates 8 or 10
has_odd = any(n % 2 == 1 for n in nums)
print(has_odd)

# Stops at 7 — short-circuits to False
all_even = all(n % 2 == 0 for n in nums)
print(all_even)

---
## 10. Comprehensions with `enumerate`, `zip`, `range`

These three built-ins appear inside comprehensions constantly. Make them muscle memory.

In [ ]:
# enumerate — pair index with value
words = ["alpha", "beta", "gamma"]
indexed = [f"{i}: {w}" for i, w in enumerate(words)]
print(indexed)

# Index-based filtering: keep items at even positions
evens_idx = [w for i, w in enumerate(words) if i % 2 == 0]
print(evens_idx)

# zip — combine parallel iterables
names = ["alice", "bob", "carol"]
ages = [30, 25, 35]
pairs = [(n, a) for n, a in zip(names, ages)]
print(pairs)

# Element-wise sum of two lists
a = [1, 2, 3, 4]
b = [10, 20, 30, 40]
sums = [x + y for x, y in zip(a, b)]
print(sums)

# range — generate a list of derived values
diagonal = [i * i for i in range(5)]
print(diagonal)

---
## 11. The Walrus Operator (`:=`) in Comprehensions

Python 3.8+ lets you assign-and-use inside an expression. Useful when you need a value **both for the filter and for the output** and don't want to compute it twice.

In [ ]:
# Without walrus — compute expensive_op TWICE
def expensive_op(x):
    return x ** 2 + 7

data = range(10)
result = [expensive_op(x) for x in data if expensive_op(x) > 20]
print(result)

# With walrus — compute once, reuse
result2 = [y for x in data if (y := expensive_op(x)) > 20]
print(result2)

**Use sparingly.** Walrus inside comprehensions is powerful but can hurt readability. Reach for it when you'd otherwise duplicate work.

---
## 12. Common Interview Patterns

These show up over and over in interviews and real backend code. Internalise them.

### 12.1 Map (transform every item)

In [ ]:
# Convert all strings to ints
strs = ["1", "2", "3", "4"]
nums = [int(s) for s in strs]
print(nums)

### 12.2 Filter (drop unwanted items)

In [ ]:
# Drop empty strings
data = ["a", "", "b", None, "c"]
clean = [x for x in data if x]
print(clean)

### 12.3 Map + Filter

In [ ]:
# Square the positives
nums = [-3, -1, 0, 2, 4, 5]
result = [n * n for n in nums if n > 0]
print(result)

### 12.4 Flatten a 2D list

In [ ]:
matrix = [[1, 2], [3, 4], [5, 6]]
flat = [x for row in matrix for x in row]
print(flat)

### 12.5 Transpose a matrix

In [ ]:
matrix = [[1, 2, 3], [4, 5, 6]]
transposed = [list(col) for col in zip(*matrix)]
print(transposed)

### 12.6 Frequency map

In [ ]:
# Count chars
s = "abracadabra"
freq = {ch: s.count(ch) for ch in set(s)}
print(freq)

### 12.7 Index of every match

In [ ]:
# Find indices of every 'a'
s = "abracadabra"
indices = [i for i, ch in enumerate(s) if ch == 'a']
print(indices)

### 12.8 Dedupe while preserving order
A list comp alone won't dedupe — use a side `seen` set or `dict.fromkeys`.

In [ ]:
# Pythonic dedupe (preserves order, Python 3.7+)
items = [3, 1, 4, 1, 5, 9, 2, 6, 5, 3, 5]
unique = list(dict.fromkeys(items))
print(unique)

### 12.9 Group / bucket by predicate
For two-way splits, two comprehensions are clean. For many-way, use a `defaultdict`.

In [ ]:
nums = range(10)
evens = [n for n in nums if n % 2 == 0]
odds  = [n for n in nums if n % 2 == 1]
print(evens, odds)

### 12.10 Project a field from a list of dicts (records)

In [ ]:
users = [
    {"id": 1, "name": "Alice", "active": True},
    {"id": 2, "name": "Bob",   "active": False},
    {"id": 3, "name": "Carol", "active": True},
]
active_names = [u["name"] for u in users if u["active"]]
print(active_names)

# Project to a smaller dict (very common in API responses)
trimmed = [{"id": u["id"], "name": u["name"]} for u in users]
print(trimmed)

### 12.11 Build a lookup index

In [ ]:
# id → record
by_id = {u["id"]: u for u in users}
print(by_id[2])

### 12.12 Pairwise / sliding pairs

In [ ]:
# Adjacent pairs (i, i+1)
nums = [10, 20, 30, 40, 50]
pairs = [(nums[i], nums[i+1]) for i in range(len(nums) - 1)]
print(pairs)

# Differences between consecutive elements
diffs = [b - a for a, b in zip(nums, nums[1:])]
print(diffs)

### 12.13 Filter rows of a 2D structure

In [ ]:
rows = [[1, 2, 3], [4, 5, 6], [7, 8, 9], [2, 2, 2]]
# Keep rows whose sum > 10
kept = [r for r in rows if sum(r) > 10]
print(kept)

### 12.14 Conditional transform of every element

In [ ]:
# Replace negatives with 0 (clamp)
xs = [-3, 4, -1, 7, -8, 2]
clamped = [x if x >= 0 else 0 for x in xs]
print(clamped)

### 12.15 Build matrix from function

In [ ]:
# Distance from origin in a grid
n = 4
grid = [[abs(r) + abs(c) for c in range(n)] for r in range(n)]
for row in grid:
    print(row)

---
## 13. Performance & Memory

- **Comprehensions beat manual `for + append`** because the append happens in C, not via attribute lookup. Expect 1.5–2x speedup.
- **Generator expressions are lazy** — use them whenever you only iterate once. Memory: O(1) instead of O(n).
- **Don't build a list just to iterate over it.** `sum([... for ...])` should be `sum(... for ...)`.
- **Avoid quadratic patterns.** `{ch: s.count(ch) for ch in s}` is O(n²) — use `collections.Counter` instead.

In [ ]:
import timeit

# Comprehension vs manual loop — comprehension wins
t1 = timeit.timeit("[x*x for x in range(1000)]", number=10_000)
t2 = timeit.timeit('''
result = []
for x in range(1000):
    result.append(x*x)
''', number=10_000)
print(f"comp: {t1:.3f}s  loop: {t2:.3f}s")

---
## 14. When NOT to Use a Comprehension

Comprehensions are great — until they aren't. Reach for a regular loop when:

1. **You need side effects** (logging, writing to a file, calling an API). Comprehensions are for *building data*, not for *doing actions*. Never write `[print(x) for x in xs]`.
2. **The logic doesn't fit on one line clearly.** If you need 3 nested `if`s and a 2-line expression, a regular loop is more readable.
3. **You need `try/except`, `break`, or `continue`.** None work inside a comprehension. (You can fake `continue` with an `if` filter; you can't fake `break`.)
4. **More than ~2 levels of nesting.** Triple-nested comprehensions are write-only code.
5. **You don't need the resulting list** — use a generator or a plain loop.

Rule of thumb: if a comprehension takes more than a couple of seconds to read, refactor it.

In [ ]:
# BAD — using a comprehension for side effects
# [print(x) for x in range(3)]   # builds a useless [None, None, None]

# GOOD — just write a loop
for x in range(3):
    print(x)

---
## 15. Gotchas & Anti-Patterns

### 15.1 Confusing `if` placement
`[x for x in xs if cond]` filters; `[a if cond else b for x in xs]` transforms. Mixing them up is the #1 list-comp bug.

### 15.2 Using `*` to multiply rows of a 2D list
Already covered in §6 — every row becomes the same object. Always build the outer dimension with a comprehension.

### 15.3 Variable leak (Python 2 only — fixed in 3)
In Python 2, comprehension variables leaked into the enclosing scope. **In Python 3 they don't.** This is fine:

```python
i = "outer"
result = [i for i in range(3)]
print(i)  # 'outer' — unchanged
```

### 15.4 Modifying the iterable while comprehending
Don't mutate the source list while iterating over it. Build a *new* list and replace.

### 15.5 Side effects inside expressions
A comprehension is for **values**, not actions. If your "expression" is `func(x)` and you don't use the result, you're abusing the syntax.

### 15.6 Quadratic comprehensions
Be mindful of nested method calls: `{ch: s.count(ch) for ch in s}` looks fine but is O(n²). Use `Counter`.

### 15.7 Over-nesting for cleverness
Triple-nested comprehensions are technically valid and almost always the wrong choice. If you can't read it in 5 seconds, neither can your interviewer.

### 15.8 Forgetting the colon for dict comps
`{x for x in xs}` is a **set**. `{x: 1 for x in xs}` is a **dict**. Watch the colon.

---
## 16. Production & Backend Use Cases

| Where you'll see them | Example |
|----------------------|---------|
| **API serialization** | `[user.to_dict() for user in users if user.active]` |
| **Filtering ORM results** | `[r.id for r in rows if r.status == "ok"]` |
| **Bulk DB inserts** | `[(u.id, u.email) for u in users]` |
| **Config parsing** | `{k: v for k, v in items if not k.startswith("_")}` |
| **Header construction** | `{k.lower(): v for k, v in headers.items()}` |
| **Test fixtures** | `[make_user(i) for i in range(100)]` |
| **Pandas-free aggregations** | `sum(row["amount"] for row in txns if row["status"] == "paid")` |
| **Data validation** | `errors = [f"{i}: {msg}" for i, msg in enumerate(results) if msg]` |

---

## Summary

- `[expr for x in iter]` — basic transform
- `[expr for x in iter if cond]` — filter
- `[a if cond else b for x in iter]` — conditional transform (every item kept)
- `[expr for x in xs for y in ys]` — nested loops (flatten / cartesian)
- `[[expr for c in cols] for r in rows]` — 2D matrix
- `{expr for x in iter}` — set comp (deduped)
- `{k: v for x in iter}` — dict comp
- `(expr for x in iter)` — generator (lazy, memory-efficient)

You now have everything you need. Open `drills.ipynb` to test your intuition, then `practice.ipynb` to lock it in.